# Multimodal Deepfake Detection Training Notebook

This notebook trains and tests the multimodal deepfake detection model using FakeAVCeleb dataset.

## Prerequisites
- Dataset copied to `FakeAVCeleb_v1.2/`
- Dependencies installed
- GPU available (RTX 4500)

In [15]:
# 1. Install dependencies (if not already done)
!pip install torch torchvision torchaudio librosa numpy scipy scikit-learn pillow opencv-python matplotlib

'pip' is not recognized as an internal or external command,
operable program or batch file.


**Note:** After running the install cell, restart the Jupyter kernel (Kernel > Restart) to ensure all packages are loaded.

## 2. Install ffmpeg (required for audio extraction)
If ffmpeg is not installed, download from https://ffmpeg.org/download.html and add to PATH, or install via Chocolatey: `choco install ffmpeg`

## 3. Prepare Dataset
Run the dataset preparation script to organize frames and audio into train/val splits.

In [16]:
# Check dataset structure
import os
print("Dataset folders:")
if os.path.exists('FakeAVCeleb_v1.2'):
    print(os.listdir('FakeAVCeleb_v1.2'))
else:
    print("Dataset not found. Please copy FakeAVCeleb_v1.2 to the project root.")

Dataset folders:
['desktop.ini', 'FakeAVCeleb_v1.2', 'frames', 'moved']


In [17]:
# Prepare dataset (subject-level split)
import sys
sys.path.append('src')
from prepare_dataset import *

# This assumes FakeAVCeleb_v1.2 is in the root
subject_split_and_process(root_videos='FakeAVCeleb_v1.2/FakeAVCeleb_v1.2', output_data='data', train_frac=0.8, fps=2, sr=16000)
print("Dataset prepared.")

Subjects found: {'real': 500, 'fake': 500}
Dataset prepared with frames and audio extracted.
Dataset prepared.


## 4. Train the Fusion Model
Train the multimodal model on the prepared dataset.

In [20]:
# Train the model
import os
print('dir exists:', os.path.exists('data/train/frames'))
print('files count:', len([f for _,_,fs in os.walk('data/train/frames') for f in fs]))
import torch 
print(torch.__version__, torch.__file__)
import sys
sys.path.append('src')
from train import train
import os

# Validate dataset paths before starting training
required_path = 'data/train/frames'
if not os.path.exists(required_path):
    raise FileNotFoundError(
        f"Required training data folder not found: {required_path}. "
        "Please run the dataset preparation cell first."
    )

config = {
    'frame_root': 'data',
    'audio_root': 'data',
    'save_dir': 'outputs/models',
    'batch_size': 16,
    'epochs': 12,
    'lr': 1e-4,
    'validate_every': 1,
    'num_workers': 4,
    'n_frames': 8
}

print('Starting training with config:', config)
train(config)
print('Training finished. Check outputs/models/fusion_best.pth')

dir exists: True
files count: 112078
2.7.1+cu118 C:\Users\Student2\AppData\Roaming\Python\Python39\site-packages\torch\__init__.py
Starting training with config: {'frame_root': 'data', 'audio_root': 'data', 'save_dir': 'outputs/models', 'batch_size': 16, 'epochs': 12, 'lr': 0.0001, 'validate_every': 1, 'num_workers': 4, 'n_frames': 8}
Epoch 1/12  Loss=0.6292  Acc=0.6715  LR=1.00e-04
Validation acc=0.9411
Saved best checkpoint: outputs/models\fusion_best.pth
Epoch 2/12  Loss=0.4239  Acc=0.8808  LR=1.00e-04
Validation acc=0.9971
Saved best checkpoint: outputs/models\fusion_best.pth
Epoch 3/12  Loss=0.3014  Acc=0.9351  LR=1.00e-04
Validation acc=0.9630
Epoch 4/12  Loss=0.2038  Acc=0.9646  LR=1.00e-04
Validation acc=0.9703
Epoch 5/12  Loss=0.1621  Acc=0.9684  LR=5.00e-05
Validation acc=0.9971
Epoch 6/12  Loss=0.0904  Acc=0.9901  LR=5.00e-05
Validation acc=0.9971
Epoch 7/12  Loss=0.0745  Acc=0.9931  LR=5.00e-05
Validation acc=0.9976
Saved best checkpoint: outputs/models\fusion_best.pth
Epoc

## 5. Evaluate the Model
Test the trained model on validation set.

In [21]:
# Evaluate
import sys
sys.path.append('src')
from evaluate import evaluate_checkpoint
import os

model_path = 'outputs/models/fusion_best.pth'
if os.path.exists(model_path):
    metrics = evaluate_checkpoint('data', model_path, split='val')
    print("Validation Metrics:", metrics)
else:
    print("Model not found. Please train the model first.")

Validation Metrics: {'accuracy': 0.9975657254138267, 'f1': 0.9987189341532154, 'auc': np.float64(0.9985286591606962), 'confusion_matrix': [[100, 0], [5, 1949]]}


## 6. Real-Time Inference Demo
Run webcam-based real-time detection.

In [22]:
# Real-time demo (press 'q' to quit)
import cv2
import sys
import os
sys.path.append('src')
from models import FusionModel
from realtime import video_stream_inference
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FusionModel(emb_dim=128).to(device)
model_path = 'outputs/models/fusion_best.pth'

if not os.path.exists(model_path):
    print('Model not found. Please train the model first.')
else:
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    video_stream_inference(model, device, source=0)


C:\Users\Student2\AppData\Roaming\Python\Python39\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Student2\AppData\Roaming\Python\Python39\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


C:\Users\Student2\AppData\Roaming\Python\Python39\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Student2\AppData\Roaming\Python\Python39\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Video source not opened: 0. Please check if webcam is available or provide a video file path.


## 7. Explainability Example
Generate Grad-CAM for a sample frame.

In [40]:
import sys
sys.path.append('src')
from explain import compute_gradcam, overlay_heatmap
from models import FusionModel
import torch
from torchvision import transforms
from PIL import Image
import cv2
import os

# Explainability sample frame
model_path = 'outputs/models/fusion_best.pth'
if not os.path.exists(model_path):
    print('Model not found. Please train the model first.')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = FusionModel(emb_dim=128).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    tf = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])

    img_paths = [
        'data/val/frames/real/id00020/00206_frame0.jpg',
        'data/val/frames/fake/id01048/00160_0_frame0.jpg'
    ]

    for img_path in img_paths:
        if os.path.exists(img_path):
            img = Image.open(img_path).convert('RGB')
            img_tensor = tf(img).unsqueeze(0).to(device)
            if img_tensor.ndim == 4:
                img_tensor = img_tensor.unsqueeze(1)
            with torch.backends.cudnn.flags(enabled=False):
                cam = compute_gradcam(model, img_tensor, target_class=1)
            heatmap = cam[0]
            frame_bgr = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
            overlay = overlay_heatmap(frame_bgr, heatmap)
            output_path = f'gradcam_overlay_{os.path.basename(img_path)}.png'
            cv2.imwrite(output_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
            print(f'Saved Grad-CAM overlay to {output_path}')
            break
    else:
        print('No sample frames found. Please prepare the dataset first.')

Saved Grad-CAM overlay to gradcam_overlay_00206_frame0.jpg.png


## Summary
This notebook implements a complete multimodal deepfake detection system:

- **Multimodal Fusion**: Combines video (MobileNetV2) and audio (CNN on mel-spectrogram) features
- **Lightweight & Real-Time**: Efficient models suitable for GPU deployment
- **Explainable**: Grad-CAM for video, audio saliency, and mismatch reasoning
- **Subject-Level Splitting**: Prevents data leakage in evaluation
- **Evaluation Metrics**: Accuracy, F1, AUC, confusion matrix
- **Single Video Classification**: Classify any provided video as real or AI-generated with explanation

**Outputs**:
- Trained model: `outputs/models/fusion_best.pth`
- Validation metrics printed
- Real-time demo (webcam)
- Grad-CAM overlay: `gradcam_overlay.png`
- Single video result with explanation

**Research Contributions**:
- Lightweight multimodal architecture for real-time deepfake detection
- Explainability techniques for audio-visual mismatch detection
- Subject-aware evaluation for better generalization

**Next Steps**:
- Test on cross-dataset (e.g., FaceForensics++)
- Optimize for mobile deployment (TensorRT)
- Add attention-based fusion for improved performance

## 8. Single Video Classification
Provide a video path to classify it as real or AI-generated with explanation.

In [41]:
# Single video classification
import sys
sys.path.append('src')
import torch  # prep torch first to avoid partial import state in nested model import
from preprocess import extract_frames, extract_audio
from models import FusionModel
from explain import compute_gradcam, overlay_heatmap
from torchvision import transforms
import librosa
import numpy as np
import cv2
from PIL import Image
import os
import tempfile
import shutil

# Debug info
print('Python:', sys.executable)
print('Torch:', torch.__version__, torch.__file__)


def classify_video(video_path, model_path='outputs/models/fusion_best.pth'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    if not os.path.exists(model_path):
        return 'Model not found. Please train the model first: ' + model_path
    
    model = FusionModel(emb_dim=128).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    with tempfile.TemporaryDirectory() as temp_dir:
        frame_dir = os.path.join(temp_dir, 'frames')
        audio_path = os.path.join(temp_dir, 'audio.wav')
        
        try:
            extract_frames(video_path, frame_dir, fps=2)
        except Exception as e:
            return f'Error extracting frames: {e}'
        
        try:
            extract_audio(video_path, audio_path, sr=16000)
        except Exception as e:
            print(f'Warning: Audio extraction failed ({e}), using dummy audio.')
        
        frame_files = sorted([os.path.join(frame_dir, f) for f in os.listdir(frame_dir) if f.endswith('.jpg')])
        if not frame_files:
            return 'No frames extracted.'
        
        tf = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
        ])
        
        frames = []
        for f in frame_files[:8]:
            img = Image.open(f).convert('RGB')
            frames.append(tf(img))
        while len(frames) < 8:
            frames.append(frames[-1])
        frame_tensor = torch.stack(frames).unsqueeze(0).to(device)
        
        if os.path.exists(audio_path):
            y, sr = librosa.load(audio_path, sr=16000)
            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, hop_length=512)
            mel = librosa.power_to_db(mel, ref=np.max)
            mel = torch.tensor(mel).unsqueeze(0).unsqueeze(0).float().to(device)
            if mel.shape[3] < 128:
                mel = torch.cat([mel, mel[:, :, :, -1:].repeat(1, 1, 1, 128 - mel.shape[3])], dim=3)
            mel = mel[:, :, :, :128]
        else:
            mel = torch.zeros(1, 1, 128, 128).to(device)
        
        with torch.no_grad():
            output, _, _ = model(frame_tensor, mel)
            probs = torch.softmax(output, dim=1)[0].cpu().numpy()
            prob_fake = float(probs[1])
            pred = 'AI-generated' if prob_fake > 0.5 else 'Real'
        
        explanation = f'The video is classified as {pred} with probability of being AI-generated: {prob_fake:.2f}.\n'
        explanation += 'High probability indicates potential deepfake characteristics in audio-visual sync or quality.' if prob_fake > 0.5 else 'Low probability suggests natural video with consistent audio-visual features.'
        
        cam = compute_gradcam(model, frame_tensor, target_class=1)
        heatmap = cam[0]
        frame_bgr = cv2.cvtColor(cv2.imread(frame_files[0]), cv2.COLOR_BGR2RGB)
        overlay = overlay_heatmap(frame_bgr, heatmap)
        cv2.imwrite('single_video_gradcam.png', cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
        explanation += '\nGrad-CAM overlay saved to single_video_gradcam.png.'
        
        return explanation

# Example usage
video_path = r'C:\Users\Student2\Desktop\deepfake_faheem\FakeAVCeleb_v1.2\FakeAVCeleb_v1.2\FakeVideo-RealAudio\Caucasian (American)\men\id00018\00181_0.mp4'
if os.path.exists(video_path):
    print(classify_video(video_path))
else:
    print('Please provide a valid video path.')

Python: c:\Program Files (x86)\Microsoft Visual Studio\Shared\Python39_64\python.exe
Torch: 2.7.1+cu118 C:\Users\Student2\AppData\Roaming\Python\Python39\site-packages\torch\__init__.py


RuntimeError: cudnn RNN backward can only be called in training mode